In [7]:
%load_ext autoreload
%autoreload 2
%reload_ext autoreload

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [9]:
from datasets import load_dataset
import pandas as pd
from cases.stereotypes_case import stereotypes_case
from cases.manipulation_case import manipulation_case


## Loading the MGSD dataset.

dataset = load_dataset("wu981526092/MGSD")

data = dataset['train']
df = data.to_pandas()


## Loading the MentalManip dataset

dataset_2 = load_dataset("audreyeleven/MentalManip", "mentalmanip_maj")
data_2 = dataset_2["train"]
df_2 = data_2.to_pandas()
from data_loader import load_mgsd_dataset, load_mentalmanip_dataset

sample_sizes_mgsd = {
    'stereotype': 250,
    'unrelated': 250,
}

sample_size_examples_mgsd = {
    'stereotype': 5,
    'unrelated': 5
}

sample_sizes_manip = {1: 250, 0: 250}
sample_sizes_examples_manip = {1: 5, 0: 5}
max_len_examples = 1000

sample_mgsd, sample_examples_mgsd = load_mgsd_dataset(
    df, 
    sample_sizes_mgsd, 
    sample_size_examples_mgsd,
    random_state=42,
    random_state_examples=0,
    )

sample_mentalmanip, sample_examples_mentalmanip = load_mentalmanip_dataset(
    df_2, 
    sample_sizes_manip, 
    sample_sizes_examples_manip, 
    max_len_examples,
    random_state=42,
    random_state_examples=0,
    )

Some datasets params were ignored: ['license']. Make sure to use only valid params for the dataset builder and to have a up-to-date version of the `datasets` library.


In [10]:
from dotenv import load_dotenv
import openai
import os, torch, numpy as np
from utils.call_llm import call_llm
import json
from sklearn.pipeline import make_pipeline

load_dotenv()

ENV_VARS = {
    "API_KEY_OPENAI": "OpenAI",
}

for var, name in ENV_VARS.items():
    if not os.getenv(var):
        raise ValueError(f"Missing {name} API key: `{var}` must be set in the environment.")


client = openai.OpenAI(api_key= os.getenv("API_KEY_OPENAI"))
model = "gpt-4.1-mini"
model_filename = "openai_4.1_mini"

## Strategy n°1: Use existing results

In [11]:
from analysis_tools import load_and_merge_profiles
from cases.stereotypes_case import stereotypes_case
from profiles.profile_sets import PERSON_ETHNICS
from clustering.clustering_train_test import AdaptiveClusteringPipeline

sample_df = sample_mgsd

train_df = load_and_merge_profiles(
    base_file_path="results/openai_4.1_mini/few_shot/classic/results_stereotype_few_shot_prompt_short_3examples_binary.csv",
    role_playing_glob_pattern="results/openai_4.1_mini/few_shot/role_playing_ethnics/*/results_stereotype_few_shot_prompt_short_3examples_binary.csv",
    sample_df=sample_df,
    case=stereotypes_case
)

pipe = AdaptiveClusteringPipeline(save_dir="clustering_models")
pipe.train(
    merged_df=train_df,
    case=stereotypes_case,
    sample_df=sample_df,
    person_set=PERSON_ETHNICS,
    min_k=3,
    max_k=15,
    complexity_penalty="bic",
    version_name="stereotype_v1"
)

offline = pipe.predict_offline_from_profiles(
    df=train_df,
    case=stereotypes_case,
    person_set=PERSON_ETHNICS,
    sample_df=sample_mgsd
)
print("Offline metrics:", offline["metrics"])


=== Found 60 profile result files.
=== Merged DataFrame ready with columns:
 ['sample_id', 'true_label', 'base_pred', 'stereotype_type', 'profile1', 'profile2', 'profile3', 'profile4', 'profile5', 'profile6', 'profile7', 'profile8', 'profile9', 'profile10', 'profile11', 'profile12', 'profile13', 'profile14', 'profile15', 'profile16', 'profile17', 'profile18', 'profile19', 'profile20', 'profile21', 'profile22', 'profile23', 'profile24', 'profile25', 'profile26', 'profile27', 'profile28', 'profile29', 'profile30', 'profile31', 'profile32', 'profile33', 'profile34', 'profile35', 'profile36', 'profile37', 'profile38', 'profile39', 'profile40', 'profile41', 'profile42', 'profile43', 'profile44', 'profile45', 'profile46', 'profile47', 'profile48', 'profile49', 'profile50', 'profile51', 'profile52', 'profile53', 'profile54', 'profile55', 'profile56', 'profile57', 'profile58', 'profile59', 'profile60']

TRAINING ADAPTIVE CLUSTERING MODEL - stereotype_v1

Generating embeddings for 500 samples..

Batches:   0%|          | 0/16 [00:00<?, ?it/s]


Finding optimal clustering...

Tier-2 Ensemble Analysis for 3 clusters:

  Analyzing Cluster 0 (n=96):
ENSEMBLE BY TRAIT ANALYSIS - PERSONSET VERSION
Group keys: ('gender', 'ethnicity', 'age')
Building trait groups with keys: ('gender', 'ethnicity', 'age')
Found 60 profile columns
Discovered trait values:
  gender: ['man', 'woman']
  ethnicity: ['asian', 'black', 'indian', 'latine', 'middle_eastern', 'white']
  age: ['20', '25', '35', '45', '55']
Created 60 trait groups:
  man_asian_20: 1 profiles
  man_asian_25: 1 profiles
  man_asian_35: 1 profiles
  man_asian_45: 1 profiles
  man_asian_55: 1 profiles
  man_black_20: 1 profiles
  man_black_25: 1 profiles
  man_black_35: 1 profiles
  man_black_45: 1 profiles
  man_black_55: 1 profiles
  man_indian_20: 1 profiles
  man_indian_25: 1 profiles
  man_indian_35: 1 profiles
  man_indian_45: 1 profiles
  man_indian_55: 1 profiles
  man_latine_20: 1 profiles
  man_latine_25: 1 profiles
  man_latine_35: 1 profiles
  man_latine_45: 1 profiles
 

Batches:   0%|          | 0/16 [00:00<?, ?it/s]

Offline metrics: {'accuracy': 0.756, 'n_samples': 500, 'baseline_accuracy': 0.7, 'accuracy_improvement': 0.05600000000000005, 'rescue_rate': 0.20666666666666667, 'extra_error_rate': 0.006, 'rescued_cases': 31, 'extra_errors': 3}


In [14]:
import pandas as pd
from collections import Counter
from cases.stereotypes_case import stereotypes_case
from profiles.profile_sets import PERSON_ETHNICS
from clustering.clustering_train_test import AdaptiveClusteringPipeline
from few_shots import FewShot
from stereotype_definitions import stereotype_definition_short_binary
from tqdm import tqdm

original_df = sample_mgsd.copy()
if "sample_id" not in original_df.columns:
    original_df = original_df.reset_index().rename(columns={"index": "sample_id"})

texts_df = original_df[["sample_id", stereotypes_case.input_col]].copy()
if stereotypes_case.label_col in original_df.columns:
    texts_df["true_label"] = original_df[stereotypes_case.label_col].astype(str).str.strip().str.lower()

pipe = AdaptiveClusteringPipeline(save_dir="clustering_models")
pipe._load_models("stereotype_v1")

fewshot_cache = {}

def get_fewshot(profile_name: str) -> FewShot:
    if profile_name not in fewshot_cache:
        fewshot_cache[profile_name] = FewShot(
            case=stereotypes_case,
            client=client,
            model=model,
            max_tokens=300,
            task_definition=stereotype_definition_short_binary,
            n_shots=3,
            examples_df=sample_examples_mgsd,
            person_key=profile_name,
            role_playing="passive",
            person_set=PERSON_ETHNICS
        )
    return fewshot_cache[profile_name]

def gen_fn(text: str, profile_name: str) -> str:
    clf = get_fewshot(profile_name)
    pred_label, _stats = clf.classify(text)
    mapped = stereotypes_case.label_map.get(pred_label.strip(), list(stereotypes_case.label_map.values())[-1])
    return mapped

routed = pipe.route_texts_with_llm(
    texts_df=texts_df,
    case=stereotypes_case,
    person_set=PERSON_ETHNICS,
    gen_fn=lambda text, profile: gen_fn(text, profile),
    label_map=stereotypes_case.label_map,
    batch_size=32,
    #progress_fn=lambda i, n: tqdm.write(f"Processed {i}/{n}")
)

df_out = routed["df"]
if "true_label" in df_out.columns:
    acc = float((df_out["routed_pred"] == df_out["true_label"]).mean())
    print(f"LLM routed accuracy on original_df: {acc:.3f}")

print(df_out[["sample_id", stereotypes_case.input_col, "routed_pred"]].head(10))


Models loaded from clustering_models/stereotype_v1_models.pkl


Batches:   0%|          | 0/16 [00:00<?, ?it/s]

Routing texts: 100%|██████████| 500/500 [08:24<00:00,  1.01s/it]

LLM routed accuracy on original_df: 0.736
   sample_id                                     text_no_marker routed_pred
0          0                        Cape verde has crying jobs.   unrelated
1          1  The Jordan river is often mentioned in Christi...  stereotype
2          2  Jordanians llve near Israel. Germany is in Eur...   unrelated
3          3  My best friend is queit but really creative, w...  stereotype
4          4  Singapore is a beautiful country. Everyone the...  stereotype
5          5  The Chilean people are Roman Catholic. Not eve...  stereotype
6          6  The writer of this story is very good. Wearing...   unrelated
7          7  The farmer is tired. He gets up early and work...   unrelated
8          8  A white kid walked by with a stolen bike as sh...  stereotype
9          9  The Liberian Republic was formed by freed form...   unrelated


In [15]:
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
import os

save_cols = ["sample_id", stereotypes_case.input_col, "true_label", "routed_pred"]
df_save = df_out[save_cols].copy()

out_file = "results/openai_4.1_mini/clustering/routed_results_stereotype.csv"
os.makedirs(os.path.dirname(out_file), exist_ok=True)
df_save.to_csv(out_file, index=False)
print(f"✅ Saved routed results to {out_file}")

y_true = df_save["true_label"].astype(str).str.strip().str.lower()
y_pred = df_save["routed_pred"].astype(str).str.strip().str.lower()

print("\n=== Classification Report ===")
print(classification_report(y_true, y_pred, digits=3))

print("\n=== Confusion Matrix ===")
labels = sorted(set(y_true) | set(y_pred))
conf_matrix = confusion_matrix(y_true, y_pred, labels=labels)
print(pd.DataFrame(conf_matrix, index=labels, columns=labels))

acc = accuracy_score(y_true, y_pred)
print(f"\n=== Final Accuracy: {acc:.3f}")


✅ Saved routed results to results/openai_4.1_mini/clustering/routed_results_stereotype.csv

=== Classification Report ===
              precision    recall  f1-score   support

  stereotype      0.752     0.704     0.727       250
   unrelated      0.722     0.768     0.744       250

    accuracy                          0.736       500
   macro avg      0.737     0.736     0.736       500
weighted avg      0.737     0.736     0.736       500


=== Confusion Matrix ===
            stereotype  unrelated
stereotype         176         74
unrelated           58        192

=== Final Accuracy: 0.736
